In [1]:
# !pip install opencv-contrib-python
# !pip install opencv-python
# !pip install ultralytics

In [2]:
import cv2
import numpy
import cut_image
from ball_detection import isOpen
import os

In [48]:
import cv2
import numpy as np


def test(img_object, img_scene):
    # paths = ["C:\\Users\\shishkin_i\\Downloads\\Telegram Desktop\\ball.jpg",
    #          "F:\\PycharmProjects\\Camera-accuracy-studies\\tmp\\hehe_tmp.png"]
    # img_object = cv2.imread(paths[0], cv2.IMREAD_COLOR)
    # img_scene = cv2.imread(paths[1], cv2.IMREAD_COLOR)

    if img_object is None or img_scene is None:
        print(" --(!) Error reading images ")
        return -1

    # Initiate ORB detector
    orb = cv2.ORB_create()

    # Find the keypoints and descriptors with ORB
    kp_object, des_object = orb.detectAndCompute(img_object, None)
    kp_scene, des_scene = orb.detectAndCompute(img_scene, None)

    # Create BFMatcher object
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)  #можно менять норму

    # Match descriptors
    matches = bf.match(des_object, des_scene)

    # Sort them in the order of their distance
    matches = sorted(matches, key=lambda x: x.distance)

    # Take the top 90% matches
    good_matches = matches[int(len(matches) * 0.1):int(len(matches) * 1)]  # что вообще происходит?

    # Draw matches
    img_matches = cv2.drawMatches(img_object, kp_object, img_scene, kp_scene,
                                  good_matches, None, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

    # Localize the object
    src_pts = np.float32([kp_object[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp_scene[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)

    # Find the Homography Matrix
    M, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)

    # Get the corners from the object image
    h, w, _ = img_object.shape
    pts = np.float32([[0, 0], [0, h - 1], [w - 1, h - 1], [w - 1, 0]]).reshape(-1, 1, 2)

    # Project corners into the scene image
    dst = cv2.perspectiveTransform(pts, M)

    # Draw the projected rectangle on the scene image
    img_scene_with_rect = cv2.polylines(img_scene.copy(), [np.int32(dst)], True, (0, 255, 0), 4)

    return img_matches, img_scene_with_rect
    # Display results
    # cv2.imshow('Good Matches & Object detection', img_matches)
    # cv2.imshow('Object detection', img_scene_with_rect)
    # cv2.waitKey(0)
    # cv2.destroyAllWindows()


if __name__ == "__main__":
    import sys



In [51]:
def generate_field(rad: int, line_width: int, height: int, width: int, channels: int = 3,
                   white: tuple = (255, 255, 255), green: tuple = (34,104,80)) -> numpy.ndarray:
    ans = numpy.zeros((height, width, channels), dtype=numpy.uint8)
    ans[::None] = green
    if rad <= 0 or line_width <= 0:
        return ans
    ans = cv2.circle(ans, (width // 2, height // 2), rad, white, thickness=line_width)
    ans = cv2.line(ans, (0, height // 2), (width, height // 2), white, thickness=line_width)
    return ans


In [ ]:
import torch

class Model:
  def __init__(self, device="cpu",):#todo rotate https://ru.wikipedia.org/wiki/%D0%9C%D0%B0%D1%82%D1%80%D0%B8%D1%86%D0%B0_%D0%BF%D0%BE%D0%B2%D0%BE%D1%80%D0%BE%D1%82%D0%B0#:~:text=%D0%9F%D0%BE%D1%81%D0%BB%D0%B5%D0%B4%D0%BE%D0%B2%D0%B0%D1%82%D0%B5%D0%BB%D1%8C%D0%BD%D1%8B%D0%B5%20%D0%BF%D0%BE%D0%B2%D0%BE%D1%80%D0%BE%D1%82%D1%8B%20%D0%BE%D0%BA%D0%BE%D0%BB%D0%BE,%D0%B4%D0%BB%D1%8F%20%D0%BC%D0%B0%D1%82%D1%80%D0%B8%D1%86%D1%8B%20%D0%BF%D0%BE%D0%B2%D0%BE%D1%80%D0%BE%D1%82%D0%B0%3A
      #todo add move
      #todo add projection
    super(Model, self).__init__()
    init_scaling_factor = torch.zeros(1, device=device)
    self.scaling_factor = torch.nn.Parameter(init_scaling_factor, requires_grad=True)
 
  def forward(self, x):
    output = self.scaling_factor * x
    return output

def minimize_diff(from_img,to_img):
    width = from_img.shape[0]
    height = from_img.shape[1]
    to_pts = [[0,0],[width,0],[0,height],[width,height]]
    from_pts = to_pts.copy()
    

In [52]:
path = '../materials_part1/5.mkv'
cam = cv2.VideoCapture(path)


def nothing(x: int):
    pass


cv2.namedWindow('test')
sum = 0
pframe = 0
isclosed = False
previous = cv2.imread("../tmp/v5_bacground.png")
cv2.createTrackbar("thickness", "test", 6, 255, nothing)
cv2.createTrackbar("radius", "test", 108, 255, nothing)
cv2.createTrackbar("greenr", "test", 34, 255, nothing)
cv2.createTrackbar("greeng", "test", 104, 255, nothing)
cv2.createTrackbar("greenb", "test", 80, 255, nothing)
while not isclosed:
    isclosed |= not isOpen("test")
    if isclosed:
        break
    # keyCode = cv2.waitKey(50)
    success, frame = cam.read()
    thic = cv2.getTrackbarPos("thickness", "test")
    rad = cv2.getTrackbarPos("radius", "test")
    greenr = cv2.getTrackbarPos("greenr", "test")
    greeng = cv2.getTrackbarPos("greeng", "test")
    greenb = cv2.getTrackbarPos("greenb", "test")
    if success == False:
        cam.release()
        cam = cv2.VideoCapture(path)
        continue
    frame = cut_image.get_cut_frame_from_frame(frame, 1)
    field = generate_field(rad, thic, frame.shape[0], frame.shape[1], green=(greenr, greeng, greenb))
    a, b = test(field, frame)
    isclosed |= not isOpen("test")
    # cv2.imshow("test", frame)
    # cv2.imshow("test", field)
    cv2.imshow("test", a)
    cv2.imshow("test2", b)
    # cv2.imshow("test", numpy.maximum(field - frame, 0))
    cv2.waitKey(1)

[[-6.97986536e-01  4.33660274e-02  2.54604012e+02]
 [-7.30693424e-01 -7.41599296e-01  3.07458344e+02]
 [-4.78103582e-03 -6.88738824e-05  1.00000000e+00]]
